# Reshaping the UN Immigration to Canada Dataset

**Course:** COMP714 Machine Learning, Module 3 Assignment

**Objective.** The United Nations publishes annual counts of immigrants to Canada by country of birth for 1980 to 2013. In its raw form the spreadsheet carries a block of descriptive rows above the table, footer notes below it, coded columns that duplicate the readable ones, and terse column names. This notebook turns that raw sheet into a tidy table with one row per country, indexed by country name, holding the continent, region, development status, one column per year, and a `Total` column.

**Approach.** Each transformation is a separate, commented step, and the notebook closes with automated checks that confirm the result has the required shape and that every total is correct. Cleaning steps are written to locate the header and the data rows from the content of the sheet rather than from fixed row numbers, so the same code works on the original Excel file and on a CSV export of the Google Sheet.

## Assignment brief

> Use the immigration data by UN. When you read it into a DataFrame, it looks like the raw sheet. Your task is to convert it into a table indexed by `Country` with the columns `Continent`, `Region`, `DevName`, one column for each year from `1980` to `2013`, and `Total`.
>
> Deliver a Jupyter Notebook and the link to a GitHub repository that hosts that notebook. Make sure the notebook is readable, that is, the repository is open to public access, all cells are executed, and you have added enough explanation and comment lines to your code.

**Author:** [Your Name] | **Instructor:** [Instructor Name] | **Date:** [Date]

### How this notebook meets each requirement

| Requirement | Where it is met |
|---|---|
| Read the UN immigration data into a DataFrame | Section 2 |
| Index by `Country` | Sections 7 and 8 |
| Columns `Continent`, `Region`, `DevName` | Sections 6 and 7 |
| One column per year, 1980 to 2013 | Sections 4 and 8 |
| `Total` column | Section 9 |
| Result matches the target table | Section 10 compares five countries with the assignment's totals |
| All cells executed | Run with *Runtime > Run all* (Colab) or *Run > Run All Cells* (Jupyter) before saving |
| Explanation and comments | A markdown note before every step and comments inside every code cell |
| Public GitHub repository | Notebook hosted in a public repository, link submitted with the notebook |

### How to run

1. Open this notebook in Google Colab (*File > Upload notebook*) or in Jupyter.
2. Click *Runtime > Run all* (Colab) or *Run > Run All Cells* (Jupyter). An internet connection is needed, because the notebook downloads the UN data itself.
3. Section 10 should print *All checks passed* and *matches* for all five countries. Save the notebook with its outputs before uploading it to GitHub.

## 1. Setup

In [1]:
# Core library for tabular data
import re
import pandas as pd

# Show every column when a wide DataFrame is printed, so the year columns are not hidden
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

print("pandas version", pd.__version__)

pandas version 2.2.3


## 2. Load the raw data

The data are the United Nations series on international migration flows to Canada, 1980 to 2013, published as the Excel workbook `Canada.xlsx`. The table sits on the worksheet *Canada by Citizenship*.

The notebook uses a local copy named `Canada.xlsx` if one is in the same folder. Otherwise it downloads the workbook from the public copy hosted for the IBM Data Visualization with Python course, so the notebook runs without any manual download. A Google Sheets share link can also be pasted into `DATA_SOURCE`.

The sheet is read **without** a header (`header=None`) because the real header row sits below about twenty rows of descriptive text. The next step finds it.

In [2]:
from pathlib import Path

# Public copy of the UN workbook (IBM Data Visualization with Python course)
UN_DATA_URL = ("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
               "IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/Canada.xlsx")

# Use a local copy if present, otherwise download the workbook
DATA_SOURCE = "Canada.xlsx" if Path("Canada.xlsx").exists() else UN_DATA_URL
SHEET = "Canada by Citizenship"   # worksheet that holds the table


def google_sheet_to_csv_url(url: str) -> str:
    """Convert a Google Sheets share link into a direct CSV export link."""
    sheet_id = re.search(r"/spreadsheets/d/([a-zA-Z0-9_-]+)", url).group(1)
    gid_match = re.search(r"[#&?]gid=(\d+)", url)
    gid = gid_match.group(1) if gid_match else "0"   # first tab by default
    return f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"


def load_raw(source: str) -> pd.DataFrame:
    """Read the sheet exactly as stored, with no header row assumed."""
    if "docs.google.com/spreadsheets" in source:
        return pd.read_csv(google_sheet_to_csv_url(source), header=None)
    if source.lower().endswith((".xlsx", ".xls")):
        book = pd.ExcelFile(source)
        # Use the named worksheet when it exists, otherwise the first one
        sheet = SHEET if SHEET in book.sheet_names else book.sheet_names[0]
        print("Reading worksheet:", sheet)
        return pd.read_excel(book, sheet_name=sheet, header=None)
    return pd.read_csv(source, header=None)


print("Data source:", DATA_SOURCE)
raw = load_raw(DATA_SOURCE)
print("Raw shape (rows, columns):", raw.shape)
raw.head(25)

Data source: https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/Canada.xlsx
Reading worksheet: Canada by Citizenship
Raw shape (rows, columns): (218, 43)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,United Nations,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Population Division,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Department of Economic and Social Affairs,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,International Migration Flows to and from Sele...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Locate the header row and keep only the data table

The original UN file starts with about twenty rows of titles and notes. Rather than hard-coding how many rows to skip, the code searches for the row that contains the column name `OdName` (the country name field) and treats it as the header. Every row above it is discarded.

In [3]:
# Find the first row in which any cell equals "OdName"
header_mask = raw.apply(lambda row: row.astype(str).str.strip().eq("OdName").any(), axis=1)
header_row = header_mask.idxmax()
print("Header found on raw row", header_row)

# Promote that row to column names and keep only the rows beneath it
df = raw.iloc[header_row + 1:].copy()
df.columns = raw.iloc[header_row].tolist()
df = df.reset_index(drop=True)
print("Columns:", list(df.columns)[:12], "...")

Header found on raw row 20
Columns: ['Type', 'Coverage', 'OdName', 'AREA', 'AreaName', 'REG', 'RegName', 'DEV', 'DevName', np.float64(1980.0), np.float64(1981.0), np.float64(1982.0)] ...


## 4. Standardise the year column labels

Excel and CSV readers may return the year headers as integers (`1980`), floats (`1980.0`) or text (`"1980"`). Mixed label types make column selection error-prone, so every year label is converted to a four-character string. String labels also match the target table and keep `df["1980"]` and `df.loc[:, "1980":"2013"]` unambiguous.

In [4]:
def normalise_label(label):
    """Turn 1980, 1980.0 or '1980' into '1980'; leave text labels unchanged (trimmed)."""
    text = str(label).strip()
    if re.fullmatch(r"\d{4}(\.0+)?", text):
        return text[:4]
    return text

df.columns = [normalise_label(c) for c in df.columns]

# The year columns are the labels made of exactly four digits, kept in time order
years = sorted(c for c in df.columns if re.fullmatch(r"\d{4}", c))
print(f"{len(years)} year columns from {years[0]} to {years[-1]}")

34 year columns from 1980 to 2013


## 5. Remove footer and blank rows

Below the table the UN file has footnote rows that contain text but no country data. A row is kept only if it names a country **and** holds at least one numeric yearly count.

In [5]:
# Convert yearly counts to numbers. Thousands separators such as "3,436" are removed first,
# and any remaining non-numeric text (footnotes) becomes NaN
df[years] = df[years].apply(
    lambda col: pd.to_numeric(col.astype(str).str.replace(",", "", regex=False).str.strip(),
                              errors="coerce"))

has_country = df["OdName"].notna() & df["OdName"].astype(str).str.strip().ne("")
has_counts = df[years].notna().any(axis=1)

rows_before = len(df)
df = df[has_country & has_counts].copy()
print(f"Dropped {rows_before - len(df)} footer or blank rows, {len(df)} country rows remain")

Dropped 0 footer or blank rows, 197 country rows remain


## 6. Drop redundant columns

`Type` and `Coverage` hold the same value on every row of this sheet, so they carry no information. `AREA`, `REG` and `DEV` are numeric codes that duplicate the readable `AreaName`, `RegName` and `DevName` columns. All five are removed. `errors="ignore"` lets the step run even if a copy of the sheet has already lost some of them.

In [6]:
# Show that Type and Coverage are constant before removing them
for col in ["Type", "Coverage"]:
    if col in df.columns:
        print(col, "unique values:", df[col].unique())

df = df.drop(columns=["Type", "Coverage", "AREA", "REG", "DEV"], errors="ignore")
print("Remaining columns:", list(df.columns[:6]), "...")

Type unique values: ['Immigrants']
Coverage unique values: ['Foreigners' 'Citizens' 'Both']
Remaining columns: ['OdName', 'AreaName', 'RegName', 'DevName', '1980', '1981'] ...


## 7. Rename columns to readable names

In [7]:
# OdName holds the country name; AreaName the continent; RegName the region
df = df.rename(columns={"OdName": "Country", "AreaName": "Continent", "RegName": "Region"})

# Trim stray spaces in the text fields so lookups such as df.loc["Albania"] work
for col in ["Country", "Continent", "Region", "DevName"]:
    df[col] = df[col].astype(str).str.strip()

## 8. Make yearly counts integers and index by country

Immigrant counts are whole numbers, so the year columns are stored as integers. Any count missing in the source is treated as zero immigrants, and the number of such cells is reported. `Country` then becomes the index, which is what the target table shows.

In [8]:
missing_counts = int(df[years].isna().sum().sum())
print("Missing yearly counts filled with 0:", missing_counts)
df[years] = df[years].fillna(0).astype(int)

df = df.set_index("Country")
df.index.name = "Country"

Missing yearly counts filled with 0: 0


## 9. Add the `Total` column

In [9]:
# Total immigrants per country across every year from 1980 to 2013
df["Total"] = df[years].sum(axis=1)

# Arrange the columns in the order required by the assignment
df = df[["Continent", "Region", "DevName"] + years + ["Total"]]
df.head()

,Continent,Region,DevName,1980,1981,1982,1983,1984,1985,1986,1987,1988,1989,1990,1991,1992,1993,1994,1995,1996,1997,1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,Total
Country,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Afghanistan,Asia,Southern Asia,Developing regions,16,39,39,47,71,340,496,741,828,1076,1028,1378,1170,713,858,1537,2212,2555,1999,2395,3326,4067,3697,3479,2978,3436,3009,2652,2111,1746,1758,2203,2635,2004,58639
Albania,Europe,Southern Europe,Developed regions,1,0,0,0,0,0,1,2,2,3,3,21,56,96,71,63,113,307,574,1264,1816,1602,1021,853,1450,1223,856,702,560,716,561,539,620,603,15699
Algeria,Africa,Northern Africa,Developing regions,80,67,71,69,63,44,69,132,242,434,491,872,795,717,595,1106,2054,1842,2292,2389,2867,3418,3406,3072,3616,3626,4807,3623,4005,5393,4752,4325,3774,4331,69439
American Samoa,Oceania,Polynesia,Developing regions,0,1,0,0,0,0,0,1,0,1,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,6
Andorra,Europe,Southern Europe,Developed regions,0,0,0,0,0,0,2,0,0,0,3,0,1,0,0,0,0,0,2,0,0,1,0,2,0,0,1,1,0,0,0,0,1,1,15


## 10. Verify the result

The checks below fail loudly if anything is wrong. They confirm the column layout, the absence of missing values and duplicate countries, the integer type of every count, and that each `Total` equals the sum of its row. The last cell compares five countries with the totals shown in the assignment's target table.

In [10]:
assert list(df.columns[:3]) == ["Continent", "Region", "DevName"], "Unexpected leading columns"
assert df.columns[-1] == "Total", "Total must be the last column"
assert df.index.name == "Country", "Index must be named Country"
assert df.index.is_unique, "Each country should appear once"
assert not df.isna().any().any(), "There should be no missing values"
assert all(pd.api.types.is_integer_dtype(df[c]) for c in years + ["Total"]), "Counts must be integers"
assert (df["Total"] == df[years].sum(axis=1)).all(), "Total must equal the sum of the years"

print("All checks passed")
print("Final shape (countries, columns):", df.shape)

All checks passed
Final shape (countries, columns): (197, 38)


In [11]:
# Totals shown in the assignment's target table
expected = {"Afghanistan": 58639, "Albania": 15699, "Algeria": 69439,
            "American Samoa": 6, "Andorra": 15}

for country, total in expected.items():
    found = df["Total"].get(country)
    status = "matches" if found == total else "DIFFERS"
    print(f"{country:<15} expected {total:>6}  found {found}  {status}")

Afghanistan     expected  58639  found 58639  matches
Albania         expected  15699  found 15699  matches
Algeria         expected  69439  found 69439  matches
American Samoa  expected      6  found 6  matches
Andorra         expected     15  found 15  matches


## 11. Final table

In [12]:
df.head()

,Continent,Region,DevName,1980,1981,1982,1983,1984,1985,1986,1987,1988,1989,1990,1991,1992,1993,1994,1995,1996,1997,1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,Total
Country,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Afghanistan,Asia,Southern Asia,Developing regions,16,39,39,47,71,340,496,741,828,1076,1028,1378,1170,713,858,1537,2212,2555,1999,2395,3326,4067,3697,3479,2978,3436,3009,2652,2111,1746,1758,2203,2635,2004,58639
Albania,Europe,Southern Europe,Developed regions,1,0,0,0,0,0,1,2,2,3,3,21,56,96,71,63,113,307,574,1264,1816,1602,1021,853,1450,1223,856,702,560,716,561,539,620,603,15699
Algeria,Africa,Northern Africa,Developing regions,80,67,71,69,63,44,69,132,242,434,491,872,795,717,595,1106,2054,1842,2292,2389,2867,3418,3406,3072,3616,3626,4807,3623,4005,5393,4752,4325,3774,4331,69439
American Samoa,Oceania,Polynesia,Developing regions,0,1,0,0,0,0,0,1,0,1,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,6
Andorra,Europe,Southern Europe,Developed regions,0,0,0,0,0,0,2,0,0,0,3,0,1,0,0,0,0,0,2,0,0,1,0,2,0,0,1,1,0,0,0,0,1,1,15


In [13]:
# Save the reshaped table for later modules
df.to_csv("canada_immigration_clean.csv")
print("Saved canada_immigration_clean.csv")

Saved canada_immigration_clean.csv


## Summary

Starting from the raw UN sheet, the notebook

1. found the true header row by content rather than by position,
2. removed the descriptive rows above the table and the footnotes below it,
3. converted year labels to consistent strings and yearly counts to integers,
4. dropped the constant `Type` and `Coverage` columns and the coded `AREA`, `REG` and `DEV` columns,
5. renamed `OdName`, `AreaName` and `RegName` to `Country`, `Continent` and `Region`,
6. indexed the table by country and added a `Total` column.

The automated checks confirm one row per country, no missing values, integer counts, and correct totals, and the final table matches the layout required by the assignment.